In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('all_idioms_similarity_combined.csv')

quality_mapping = {
    'Good Translation': 5,
    'Unnatural': 4,
    'Partial Translation': 3,
    'Literal Translation': 2,
    'Mistranslation': 1,
    'Hallucination': 0
}

df['quality_score'] = df['eval_category'].map(quality_mapping)

print("\n" + "="*80)
print("IDIOM TRANSLATION QUALITY ANALYSIS")
print("="*80)

# Overall dataset metrics
total_samples = len(df)
unique_idioms = df['idiom'].nunique()
avg_quality = df['quality_score'].mean()
good_pct = (df['quality_score'] >= 4).sum() / total_samples * 100
problem_pct = (df['quality_score'] <= 2).sum() / total_samples * 100

print(f"\nOVERALL DATASET")
print(f"   Total sentences: {total_samples}")
print(f"   Unique idioms: {unique_idioms}")
print(f"   Mean quality: {avg_quality:.2f}/5.0")
print(f"   Good/Acceptable: {good_pct:.1f}%")
print(f"   Problematic: {problem_pct:.1f}%")

# Category breakdown with stats
print(f"\nCATEGORY STATISTICS")
category_stats = df.groupby('eval_category').agg({
    'quality_score': ['count', 'mean'],
    'similarity_gap': 'mean'
}).round(3)
category_stats.columns = ['count', 'avg_quality', 'avg_gap']
category_stats['percentage'] = (category_stats['count'] / total_samples * 100).round(1)
category_stats = category_stats[['count', 'percentage', 'avg_gap']]
category_stats = category_stats.sort_values('count', ascending=False)
print(category_stats)

# Similarity gap by category
print(f"\nSIMILARITY GAP ANALYSIS BY CATEGORY")
gap_analysis = df.groupby('eval_category')['similarity_gap'].agg([
    'mean', 'std', 'count'
]).round(4)
gap_analysis = gap_analysis.sort_values('mean')
print(gap_analysis)

# Key findings - per idiom with n >= 10
print(f"\n" + "="*80)
print("KEY FINDINGS (Idioms with n >= 10)")
print("="*80)

idiom_stats = df.groupby('idiom').agg({
    'quality_score': ['mean', 'count'],
    'similarity_gap': 'mean'
}).round(3)
idiom_stats.columns = ['_'.join(col).strip() for col in idiom_stats.columns.values]
idiom_stats_filtered = idiom_stats[idiom_stats['quality_score_count'] >= 10].sort_values('quality_score_mean', ascending=False)

if len(idiom_stats_filtered) > 0:
    best_idioms = idiom_stats_filtered.nlargest(10, 'quality_score_mean')
    print(f"\nBEST PERFORMING:")
    for idiom, row in best_idioms.iterrows():
        print(f"   {idiom}: {row['quality_score_mean']:.2f}/5.0 (n={int(row['quality_score_count'])})")

    worst_idioms = idiom_stats_filtered.nsmallest(10, 'quality_score_mean')
    print(f"\nWORST PERFORMING:")
    for idiom, row in worst_idioms.iterrows():
        print(f"   {idiom}: {row['quality_score_mean']:.2f}/5.0 (n={int(row['quality_score_count'])})")

print(f"\nSIMILARITY GAP INSIGHTS:")
print(f"   Well-naturalized (gap > -0.05): {(df['similarity_gap'] > -0.05).sum()} ({(df['similarity_gap'] > -0.05).sum()/total_samples*100:.1f}%)")
print(f"   Acceptable (-0.20 < gap < -0.05): {((df['similarity_gap'] <= -0.05) & (df['similarity_gap'] > -0.20)).sum()} ({((df['similarity_gap'] <= -0.05) & (df['similarity_gap'] > -0.20)).sum()/total_samples*100:.1f}%)")
print(f"   Too literal (gap < -0.20): {(df['similarity_gap'] < -0.20).sum()} ({(df['similarity_gap'] < -0.20).sum()/total_samples*100:.1f}%)")


IDIOM TRANSLATION QUALITY ANALYSIS

OVERALL DATASET
   Total sentences: 6782
   Unique idioms: 181
   Mean quality: 4.31/5.0
   Good/Acceptable: 80.3%
   Problematic: 15.2%

CATEGORY STATISTICS
                     count  percentage  avg_gap
eval_category                                  
Good Translation      4710        69.4   -0.073
Literal Translation    787        11.6   -0.170
Unnatural              738        10.9   -0.074
Partial Translation    300         4.4   -0.076
Mistranslation         245         3.6   -0.125
Hallucination            2         0.0    0.116

SIMILARITY GAP ANALYSIS BY CATEGORY
                       mean     std  count
eval_category                             
Literal Translation -0.1695  0.1322    787
Mistranslation      -0.1250  0.1547    245
Partial Translation -0.0762  0.1023    300
Unnatural           -0.0742  0.1051    738
Good Translation    -0.0732  0.0959   4710
Hallucination        0.1156  0.0439      2

KEY FINDINGS (Idioms with n >= 10)

BES